### Collect zotero entries and their attachments, store the results in a file

**Warning**: misses most of what's in my Zotero DB.  
            *Maybe it's only getting entries I made since switching to zotero 7?*

In [1]:
%load_ext autoreload
%autoreload 2

from icecream import ic
import pathlib as pl
from collections.abc import Iterable
from pyzotero import zotero
from collections import defaultdict
import pandas as pd
import refwrangle as rfw
import matplotlib.pyplot as plt

In [2]:
def ensure_iterable(obj):
    """Wraps a scalar in a list if it's not iterable."""
    if isinstance(obj, str):  # Strings are technically iterable but should remain scalars
        return [obj]
    elif isinstance(obj, Iterable):
        return obj
    else:
        return [obj]
    
def get_first_creator(item):
    """Get first creator (usually author) from zotero db top level parent item"""
    # Check if creators key exists and is not empty
    if 'creators' in item['data'] and item['data']['creators']:
        creators = item['data']['creators']
        for creator in creators:
            if creator['creatorType'] == 'author':
                if 'name' in creator:
                    return creator['name']
                else:
                    return f"{creator['lastName']}, {creator['firstName']}"
    return ''  # no creators found

def get_item_venue(item):
    """Get the venue where item appeared.  Zotero puts this in many different fields"""
    data = item['data']
    
    # Check different possible venue fields in order of priority: I know these exist
    venueKeyInPriority = [
        'publicationTitle',  # For journal articles
        'journalAbbreviation', # For journal articles
        'bookTitle',         # For book chapters
        'publisher',         # For books
        'proceedingsTitle',  # For conference papers
        'blogTitle',         # For blog posts
        'websiteTitle',      # For web pages
        'encyclopediaTitle', # For encyclopedia articles
        'dictionaryTitle',   # For dictionary entries
        'conferenceName',    # For conference papers
        'university',        # For theses
        'publisher',         # For books
        'institution',       # For reports
        'libraryCatalog',    # For library catalog entries (how zotero files YouTube)
        'place',             # For location
    ]
    
    for field in venueKeyInPriority:
        if field in data and data[field]:
            return data[field] # assume field is the venue key
        
    # If here, didn't find any of the priority venues.
    # Try to find one in this possibly ficticious dict from perplexity
    # https://www.perplexity.ai/search/for-the-item-type-forum-post-w-_4Myygs7Qni_.0iVwQ3vLg#3

    venueKeyForItemType = {
        'blogPost': 'blogTitle',
        'book': 'publisher',
        'bookSection': 'bookTitle',
        'computerProgram': 'company',
        'conferencePaper': 'proceedingsTitle',
        'dataset': 'repository',
        'dictionaryEntry': 'dictionaryTitle',
        'document': 'archive',
        'email': 'subject',
        'encyclopediaArticle': 'encyclopediaTitle',
        'forumPost': 'forumTitle',
        'journalArticle': 'publicationTitle',
        'magazineArticle': 'publicationTitle',
        'manuscript': 'archive',
        'newspaperArticle': 'publicationTitle',
        'note': 'note',
        'preprint': 'repository',
        'presentation': 'conferenceName',
        'report': 'institution',
        'thesis': 'university',
        'videoRecording': 'libraryCatalog',
        'webpage': 'websiteTitle'
    }

    try:
        itemType = data['itemType']
        return venueKeyForItemType[itemType]
    except:
        print(f"failed to find venue for {rfw.get_citation_key(data)}")
        return ''

def get_parent_metadata(parent_item, collection_names):
    """Parse a parent_item's metadata into a dict w/ standardized names in the keys"""

    # Get "author": can be many things in zotero
    pdat = parent_item['data']
    firstCreator = ''
    if 'creators' in pdat and pdat['creators']:
        creators = pdat['creators']
        ctypes = [creator['creatorType'] for creator in creators]
        hasAuthor = 'author' in ctypes # so can prioritize author creator
        for creator in creators:
            if (creator['creatorType'] == 'author') or not hasAuthor:
                if 'name' in creator:
                    firstCreator = creator['name']
                else:
                    firstCreator = f"{creator['lastName']}, {creator['firstName']}"
                break

    def get_if_there(pkey):
        return pdat[pkey] if pkey in pdat else ''

    return dict(parentFirstCreator = firstCreator,
                # convert collections from zotero keys to names
                parentCollections=[collection_names[colkey] for colkey in pdat['collections']],
                
                parentCitekey=rfw.get_citation_key(pdat),
                parentVenue=get_item_venue(parent_item),
                parentDate = get_if_there('date'),
                parentTitle = get_if_there('title'),
                parentURL=get_if_there('url'),
                parentZotkey=pdat['key'])

In [5]:
# Read and parse the zotero database

# Remote API access
zot = zotero.Zotero(rfw.library_id, rfw.library_type, rfw.api_key)

# Local API access: never worked
# api_key_local_and_inline = 'auDi25kN9dMPuWyhn6qfl7dM'  
#zot = zotero.Zotero(rfw.library_id, "user", api_key_local_and_inline, local=True)  # <-- Added `local=True`
#
#zot.endpoint = "http://localhost:23119/api/"


collection_names = (defaultdict(str) # handle missed collection (W5HNMQSX for parent QTESUD23)
                    | {collection['key']: collection['data']['name'] for collection in zot.collections()})

# Get all pdf and html attachments and associate them with their parent info
# parentItems = zot.everything(zot.top())

zotero_cache = rfw.ZoteroCache()
parentItems = zotero_cache.get_data()

Cache is outdated or missing. Fetching fresh data.
/users/60638/items/top?limit=100&start=100
/users/60638/items/top?limit=100&start=200
/users/60638/items/top?limit=100&start=300
/users/60638/items/top?limit=100&start=400
/users/60638/items/top?limit=100&start=500
/users/60638/items/top?limit=100&start=600
/users/60638/items/top?limit=100&start=700
/users/60638/items/top?limit=100&start=800
/users/60638/items/top?limit=100&start=900
/users/60638/items/top?limit=100&start=1000
/users/60638/items/top?limit=100&start=1100
/users/60638/items/top?limit=100&start=1200
/users/60638/items/top?limit=100&start=1300
/users/60638/items/top?limit=100&start=1400
/users/60638/items/top?limit=100&start=1500
/users/60638/items/top?limit=100&start=1600


In [6]:
child_exceptions, attachment_files = [], []
for parent in parentItems:
    if len(parent['meta']) < 1:
        continue # skip standalone notes or entries e.g. topictags.org

    pdat_always_save = get_parent_metadata(parent, collection_names)

    if rfw.is_youtube_video(parent):
        sourceInfo = pdat_always_save.copy()
        sourceInfo['contentType'] = 'youtube_video'
        attachment_files.append(sourceInfo) # not truly a file: info comes from URL
        continue
    
    parentCitekey = pdat_always_save['parentCitekey']
    errorParentIDstr = f'[{parentCitekey}]: {pdat_always_save['parentTitle']}'
    for child in zot.children(parent['key']):
        if rfw.is_ignorable_child(child):
            continue

        cdat = child['data'] | pdat_always_save
        fixed = {'Fixed':False}
        try:
            cdat['file_basename'] = cdat['path'].removeprefix("attachments:")
            guessFNm = rfw.lit_attachment_dir_shared / cdat['file_basename']
            if guessFNm.exists():
                cdat['file_fullpath'] = guessFNm
            else:
                errStr = f'Error for {errorParentIDstr}. Full path does not exist: "{guessFNm}"'
                print(errStr)
                child_exceptions.append({'exception':errStr} | cdat | fixed)
        except Exception as e:
            print(f'Error for  {errorParentIDstr}: {e}')
            # No idea why these errors occur.  Try to fix
            for ext in ['pdf', 'html']: 
                guessBasename = f'{parentCitekey}.{ext}'
                guessFNm = rfw.lit_attachment_dir_shared / guessBasename
                if guessFNm.exists():
                    cdat['file_basename'] = guessBasename
                    cdat['file_fullpath'] = guessFNm
                    break # if find pdf first, don't get html
            if  'file_basename' in cdat:
                print(f"\tFix by guess basename worked: {cdat['file_basename']}")
                fixed = {'Fixed':True}
            else:
                print(f'\tCould not fix it. Full path does not exist: "{guessFNm}"')
                fixed = {'Fixed':False}
                continue # no html or pdf: don't allow it in attachment_files (below)

            child_exceptions.append({'exception': str(e)} | cdat | fixed)

        attachment_files.append(cdat)

attachment_files = pd.DataFrame(attachment_files)
child_exceptions = pd.DataFrame(child_exceptions)

Error for [Liu24canLLMstatCausalRsn]: Are LLMs Capable of Data-based Statistical and Causal Reasoning? Benchmarking Advanced Quantitative Reasoning with Data. Full path does not exist: "C:\Users\scott\OneDrive\share\ref\zotero\Liu24canLLMstatCausalRsn.pdf"
Error for [Chen24areOpenSrcLLMcatchUp]: ChatGPT's One-year Anniversary: Are Open-Source Large Language Models Catching up?. Full path does not exist: "C:\Users\scott\OneDrive\share\ref\zotero\Chen24areOpenSrcLLMcatchUp.pdf"
Error for [Nelson23ancillarySat_CAISO_ERCOT]: Ancillary Market Saturation in CAISO and ERCOT: A Series of Predictable Events. Full path does not exist: "C:\Users\scott\OneDrive\share\ref\obsidian\Obsidian Share Vault\lit\lit_sources\Nelso23ancillarySat_CAISO_ERCOT.pdf"
Error for [Loutan17reliabSrvcPV300MW]: Demonstration of essential reliability services by a 300-MW solar photovoltaic power plant. Full path does not exist: "C:\Users\scott\Library\CloudStorage\OneDrive-Personal\share\ref\zotero\Loutan17reliabSrvcPV

In [7]:
unfixed = child_exceptions.query('Fixed != True')
if (nUnfixed := len(unfixed)) > 0:
    print(f'{nUnfixed} unfixed child exceptions')
    display(unfixed)
else:
    print(f'Fixed {child_exceptions.Fixed.value_counts().values[0]} of {len(child_exceptions)} exceptions')

61 unfixed child exceptions


,exception,key,version,parentItem,itemType,linkMode,title,accessDate,url,note,...,parentFirstCreator,parentCollections,parentCitekey,parentVenue,parentDate,parentTitle,parentURL,parentZotkey,file_basename,Fixed
0,Error for [Liu24canLLMstatCausalRsn]: Are LLMs...,6DG4QZIK,11166,YVYVG7FM,attachment,linked_file,Liu24canLLMstatCausalRsn.pdf,,,"<p xmlns=""http://www.w3.org/1999/xhtml"" id=""ti...",...,"Liu, Xiao",[Generative AI],Liu24canLLMstatCausalRsn,arXiv.org,2024-02-27,Are LLMs Capable of Data-based Statistical and...,http://arxiv.org/abs/2402.17644,YVYVG7FM,C:\Users\scott\OneDrive\share\ref\zotero\Liu24...,False
1,Error for [Chen24areOpenSrcLLMcatchUp]: ChatGP...,9R9TEELT,11132,8VJN6W8W,attachment,linked_file,Chen24areOpenSrcLLMcatchUp.pdf,,,,...,"Chen, Hailin",[Generative AI],Chen24areOpenSrcLLMcatchUp,arXiv.org,2024-01-15,ChatGPT's One-year Anniversary: Are Open-Sourc...,http://arxiv.org/abs/2311.16989,8VJN6W8W,C:\Users\scott\OneDrive\share\ref\zotero\Chen2...,False
2,Error for [Nelson23ancillarySat_CAISO_ERCOT]: ...,K7QSC288,14688,3P9JSVPR,attachment,linked_file,Nelson23ancillarySat_CAISO_ERCOT.pdf,,,,...,"Nelson, Brent",[CAISO Market],Nelson23ancillarySat_CAISO_ERCOT,Ascend Analytics,7/23,Ancillary Market Saturation in CAISO and ERCOT...,https://www.ascendanalytics.com/blog/ancillary...,3P9JSVPR,Nelso23ancillarySat_CAISO_ERCOT.pdf,False
3,Error for [Loutan17reliabSrvcPV300MW]: Demonst...,66FLZTYY,14363,9KIIN5IG,attachment,linked_file,Loutan17reliabSrvcPV300MW.pdf,,,"<p xmlns=""http://www.w3.org/1999/xhtml"" id=""ti...",...,"Loutan, Clyde",[Forecast PV],Loutan17reliabSrvcPV300MW,"National Renewable Energy Lab.(NREL), Golden, ...",2017,Demonstration of essential reliability service...,https://www.osti.gov/biblio/1349211,9KIIN5IG,/Users/scott/Library/CloudStorage/OneDrive-Per...,False
4,Error for [OpenAI24synthVoiceChllngOppty]: Nav...,YJM5A4WC,11151,CANXWGAA,attachment,linked_file,OpenAI24synthVoiceChllngOppty.pdf,,,,...,OpenAI,[Generative AI],OpenAI24synthVoiceChllngOppty,websiteTitle,"March 29, 2024",Navigating the Challenges and Opportunities of...,https://openai.com/blog/navigating-the-challen...,CANXWGAA,C:\Users\scott\OneDrive\share\ref\zotero\OpenA...,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,Error for [Bidgely19amInsightsRprt]: AMI-Drive...,JIPHE93A,14479,EKLZCYP5,attachment,linked_file,Bidgely19amInsightsRprt.pdf,,,,...,"Bidgely,",[Forecast aggr_disaggr],Bidgely19amInsightsRprt,"Bidgely, inc.",August 2019,AMI-Driven Insights Report,https://www.idcutilitiessummit.com/index/RESOU...,EKLZCYP5,C:\Users\scott\OneDrive\share\ref\zotero\paper...,False
57,Error for [Mayhorn16disaggLdRealWrldPerf]: Loa...,62I2GBJK,14480,RCMMDNV6,attachment,linked_file,Mayhorn16disaggLdRealWrldPerf.pdf,,,,...,"Mayhorn, Ebony T",[Forecast aggr_disaggr],Mayhorn16disaggLdRealWrldPerf,ACEEE Summer Study on Energy Efficiency in Bui...,2016,Load Disaggregation Technologies: Real World a...,,RCMMDNV6,C:\Users\scott\OneDrive\share\ref\zotero\paper...,False
58,Error for [Hare18disaggHmLdDmdResp]: Disaggreg...,SX6AFLGY,14480,WA8IQAXP,attachment,linked_file,Hare18disaggHmLdDmdResp.pdf,,,,...,"Hare, Jeremy (Jeremy B. )",[Forecast aggr_disaggr],Hare18disaggHmLdDmdResp,Massachusetts Institute of Technology,2018,Disaggregation of residential home energy via ...,https://dspace.mit.edu/handle/1721.1/117983,WA8IQAXP,C:\Users\scott\OneDrive\share\ref\zotero\paper...,False
59,Error for [Rehman21LoadDisaggThesis]: Load Dis...,9G5G5RPX,14481,8MAAZN8P,attachment,linked_file,Rehman21LoadDisaggThesis.pdf,,,,...,"Rehman, Attique Ur",[Forecast aggr_disaggr],Rehman21LoadDisaggThesis,Auckland University of Technology,2021,Load Disaggregation: Towards Energy Efficient ...,https://openrepository.aut.ac.nz/handle/10292/...,8MAAZN8P,C:\Users\scott\OneDrive\share\ref\zotero\paper...,False


In [8]:
# Find entries with unclassified venu

unclassifiedVenues = []
for parent in parentItems:
    pinfo = get_parent_metadata(parent, collection_names)
    if len(pinfo['parentVenue'])<1:
        #print('Unclassified Venue:')
        #display(pinfo)
        unclassifiedVenues.append(pinfo)

if (nUnclassifVenues := len(unclassifiedVenues)) > 0:
    print(f'There were {nUnclassifVenues} unclassified venues:')
    unclassifiedVenues = pd.DataFrame(unclassifiedVenues)
    display(unclassifiedVenues)
else:
    print('No unclassified venues')

No unclassified venues


In [9]:
# Count number of attachments (not parents) per collection
citekeysInCollection = defaultdict(list)
for row in attachment_files.itertuples(index=False):
    for collection in row.parentCollections:
        try:
            citekeysInCollection[collection].append(row.parentCitekey)
        except Exception as e:
            ic(e, row)

collection_counts = pd.Series({collection: len(filekeys) for collection, filekeys in citekeysInCollection.items()})
collection_counts.sort_values(ascending=False, inplace=True)
# plt.figure(figsize=(5, 10))  # Width is set to 8 inches, height to 5 inches
# collection_counts.plot(kind='barh',)

In [10]:
rfw.save_pickle_data(rfw.extractedZoteroEntriesFNm, 
                 {'attachment_files':attachment_files, 
                 'child_exceptions':child_exceptions,
                 'collection_counts':collection_counts})

Writing to C:\Users\scott\OneDrive\share\ref\refwrangle\dat\zotero_entries.pkl...


In [11]:
child_exceptions

,exception,key,version,parentItem,itemType,linkMode,title,accessDate,url,note,...,parentFirstCreator,parentCollections,parentCitekey,parentVenue,parentDate,parentTitle,parentURL,parentZotkey,file_basename,Fixed
0,Error for [Liu24canLLMstatCausalRsn]: Are LLMs...,6DG4QZIK,11166,YVYVG7FM,attachment,linked_file,Liu24canLLMstatCausalRsn.pdf,,,"<p xmlns=""http://www.w3.org/1999/xhtml"" id=""ti...",...,"Liu, Xiao",[Generative AI],Liu24canLLMstatCausalRsn,arXiv.org,2024-02-27,Are LLMs Capable of Data-based Statistical and...,http://arxiv.org/abs/2402.17644,YVYVG7FM,C:\Users\scott\OneDrive\share\ref\zotero\Liu24...,False
1,Error for [Chen24areOpenSrcLLMcatchUp]: ChatGP...,9R9TEELT,11132,8VJN6W8W,attachment,linked_file,Chen24areOpenSrcLLMcatchUp.pdf,,,,...,"Chen, Hailin",[Generative AI],Chen24areOpenSrcLLMcatchUp,arXiv.org,2024-01-15,ChatGPT's One-year Anniversary: Are Open-Sourc...,http://arxiv.org/abs/2311.16989,8VJN6W8W,C:\Users\scott\OneDrive\share\ref\zotero\Chen2...,False
2,Error for [Nelson23ancillarySat_CAISO_ERCOT]: ...,K7QSC288,14688,3P9JSVPR,attachment,linked_file,Nelson23ancillarySat_CAISO_ERCOT.pdf,,,,...,"Nelson, Brent",[CAISO Market],Nelson23ancillarySat_CAISO_ERCOT,Ascend Analytics,7/23,Ancillary Market Saturation in CAISO and ERCOT...,https://www.ascendanalytics.com/blog/ancillary...,3P9JSVPR,Nelso23ancillarySat_CAISO_ERCOT.pdf,False
3,Error for [Loutan17reliabSrvcPV300MW]: Demonst...,66FLZTYY,14363,9KIIN5IG,attachment,linked_file,Loutan17reliabSrvcPV300MW.pdf,,,"<p xmlns=""http://www.w3.org/1999/xhtml"" id=""ti...",...,"Loutan, Clyde",[Forecast PV],Loutan17reliabSrvcPV300MW,"National Renewable Energy Lab.(NREL), Golden, ...",2017,Demonstration of essential reliability service...,https://www.osti.gov/biblio/1349211,9KIIN5IG,/Users/scott/Library/CloudStorage/OneDrive-Per...,False
4,Error for [OpenAI24synthVoiceChllngOppty]: Nav...,YJM5A4WC,11151,CANXWGAA,attachment,linked_file,OpenAI24synthVoiceChllngOppty.pdf,,,,...,OpenAI,[Generative AI],OpenAI24synthVoiceChllngOppty,websiteTitle,"March 29, 2024",Navigating the Challenges and Opportunities of...,https://openai.com/blog/navigating-the-challen...,CANXWGAA,C:\Users\scott\OneDrive\share\ref\zotero\OpenA...,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,Error for [Bidgely19amInsightsRprt]: AMI-Drive...,JIPHE93A,14479,EKLZCYP5,attachment,linked_file,Bidgely19amInsightsRprt.pdf,,,,...,"Bidgely,",[Forecast aggr_disaggr],Bidgely19amInsightsRprt,"Bidgely, inc.",August 2019,AMI-Driven Insights Report,https://www.idcutilitiessummit.com/index/RESOU...,EKLZCYP5,C:\Users\scott\OneDrive\share\ref\zotero\paper...,False
57,Error for [Mayhorn16disaggLdRealWrldPerf]: Loa...,62I2GBJK,14480,RCMMDNV6,attachment,linked_file,Mayhorn16disaggLdRealWrldPerf.pdf,,,,...,"Mayhorn, Ebony T",[Forecast aggr_disaggr],Mayhorn16disaggLdRealWrldPerf,ACEEE Summer Study on Energy Efficiency in Bui...,2016,Load Disaggregation Technologies: Real World a...,,RCMMDNV6,C:\Users\scott\OneDrive\share\ref\zotero\paper...,False
58,Error for [Hare18disaggHmLdDmdResp]: Disaggreg...,SX6AFLGY,14480,WA8IQAXP,attachment,linked_file,Hare18disaggHmLdDmdResp.pdf,,,,...,"Hare, Jeremy (Jeremy B. )",[Forecast aggr_disaggr],Hare18disaggHmLdDmdResp,Massachusetts Institute of Technology,2018,Disaggregation of residential home energy via ...,https://dspace.mit.edu/handle/1721.1/117983,WA8IQAXP,C:\Users\scott\OneDrive\share\ref\zotero\paper...,False
59,Error for [Rehman21LoadDisaggThesis]: Load Dis...,9G5G5RPX,14481,8MAAZN8P,attachment,linked_file,Rehman21LoadDisaggThesis.pdf,,,,...,"Rehman, Attique Ur",[Forecast aggr_disaggr],Rehman21LoadDisaggThesis,Auckland University of Technology,2021,Load Disaggregation: Towards Energy Efficient ...,https://openrepository.aut.ac.nz/handle/10292/...,8MAAZN8P,C:\Users\scott\OneDrive\share\ref\zotero\paper...,False


In [12]:
collection_counts[collection_counts> 6]

Generative AI                           306
                                        282
Hot Takes US Elect 2024                 276
NeuroPsychoLinguisticPolitics           128
MediaAdsPolit                           118
PoliticalML                              64
Battery Review                           61
priceFrcstAEMO                           53
IdentityPolitics                         52
Conformal Prediction                     45
PowerMarkets                             43
PollMethods                              38
battFires                                37
Voting Systems                           30
Forecast aggr_disaggr                    28
ElectionPredFeats                        28
MisDisinformation                        26
CAISO Market                             25
Polarization                             25
copula                                   21
FocusGroups                              20
Contextual Optimization                  18
Optimization                    